# 01 — Plate Metadata

Define and save the plate layout configuration used throughout the pipeline.
Run this notebook first before any other step.

In [1]:
import json
import os

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/results', exist_ok=True)

## Define plate layout

In [2]:
rows = list('ABCDEFGHIJKLMNOP')  # 16 rows
cols = list(range(1, 25))        # 24 columns

# Control well definitions
# Low controls:  Column 1 (all rows) + Column 23 rows A-H
# Medium controls: Column 2 rows A-H + Column 23 rows I-P
# High controls: Column 2 rows I-P + Column 24 (all rows)

def get_well_type(row, col):
    row_idx = rows.index(row)
    if col == 1:
        return 'Low'
    elif col == 2 and row_idx < 8:
        return 'Medium'
    elif col == 2 and row_idx >= 8:
        return 'High'
    elif col == 23 and row_idx < 8:
        return 'Low'
    elif col == 23 and row_idx >= 8:
        return 'Medium'
    elif col == 24:
        return 'High'
    else:
        return 'Experimental'

well_types = {}
for row in rows:
    for col in cols:
        well = f'{row}{col:02d}'
        well_types[well] = get_well_type(row, col)

low_wells  = [w for w, t in well_types.items() if t == 'Low']
med_wells  = [w for w, t in well_types.items() if t == 'Medium']
high_wells = [w for w, t in well_types.items() if t == 'High']
exp_wells  = [w for w, t in well_types.items() if t == 'Experimental']

print(f'Low control wells  : {len(low_wells)}')
print(f'Medium control wells: {len(med_wells)}')
print(f'High control wells : {len(high_wells)}')
print(f'Experimental wells : {len(exp_wells)}')

Low control wells  : 24
Medium control wells: 16
High control wells : 24
Experimental wells : 320


## Save metadata to JSON

In [3]:
metadata = {
    'assay_name': 'HTS Fluorescence Assay',
    'units': 'RFU',
    'plate_format': '384-well',
    'rows': rows,
    'columns': cols,
    'num_plates': 5,
    'plate_files': [
        'sample_384_plate.csv',
        'sample_384_plate2.csv',
        'sample_384_plate3.csv',
        'sample_384_plate4.csv',
        'sample_384_plate5.csv'
    ],
    'well_types': well_types,
    'low_wells': low_wells,
    'medium_wells': med_wells,
    'high_wells': high_wells,
    'experimental_wells': exp_wells,
    'control_rfu': {
        'low':  {'mean': 300,   'std': 80,   'min': 50,    'max': 600},
        'medium': {'mean': 10000, 'std': 2000, 'min': 4000,  'max': 18000},
        'high': {'mean': 30000, 'std': 2000, 'min': 22000, 'max': 38000}
    }
}

with open('data/plate_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: data/plate_metadata.json')
print(f"Plates: {metadata['num_plates']}")
print(f"Format: {metadata['plate_format']}")

Saved: data/plate_metadata.json
Plates: 5
Format: 384-well
